In [1]:
# Cell 1 — Load and inspect
import pandas as pd
df = pd.read_parquet("../data/raw/qualifying_laps_raw.parquet")
print(df.shape)
df.head()

(39300, 41)


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Year,RoundNumber,EventName,Country,CircuitShortName,TrackTemp,AirTemp,Humidity,WindSpeed,Rainfall
0,0 days 00:24:04.661000,VER,33,NaT,1.0,1.0,0 days 00:21:18.434000,NaT,NaT,0 days 00:01:24.332000,...,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,33.84359,29.060256,46.544872,0.966667,False
1,0 days 00:25:35.160000,VER,33,0 days 00:01:30.499000,2.0,1.0,NaT,NaT,0 days 00:00:28.807000,0 days 00:00:38.980000,...,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,33.84359,29.060256,46.544872,0.966667,False
2,0 days 00:27:25.520000,VER,33,0 days 00:01:50.360000,3.0,1.0,NaT,0 days 00:27:23.844000,0 days 00:00:34.462000,0 days 00:00:43.825000,...,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,33.84359,29.060256,46.544872,0.966667,False
3,0 days 00:44:22.361000,VER,33,NaT,4.0,2.0,0 days 00:42:21.384000,NaT,NaT,0 days 00:00:54.163000,...,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,33.84359,29.060256,46.544872,0.966667,False
4,0 days 00:45:52.679000,VER,33,0 days 00:01:30.318000,5.0,2.0,NaT,NaT,0 days 00:00:28.964000,0 days 00:00:38.858000,...,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,33.84359,29.060256,46.544872,0.966667,False


In [2]:
# Cell 2 — Check coverage
print("Seasons:", sorted(df["Year"].unique()))
print("Circuits:", df["EventName"].nunique())
print("Drivers:", df["Driver"].nunique())

Seasons: [2021, 2022, 2023, 2024, 2025, 2026]
Circuits: 30
Drivers: 36


In [3]:
# Cell 3 — Check weather columns
df[["TrackTemp", "AirTemp", "Humidity", "WindSpeed", "Rainfall"]].describe()

,TrackTemp,AirTemp,Humidity,WindSpeed
count,39300.000000,39300.000000,39300.000000,39300.000000
mean,34.293738,23.350695,54.704914,1.586800
std,9.932483,5.541847,19.154925,0.949822
min,12.549412,11.728235,12.531646,0.197436
25%,26.889773,19.164198,38.653333,0.841176
50%,34.272632,24.188000,55.807229,1.439241
75%,41.098810,27.024359,68.355263,2.185841
max,59.239080,35.460759,93.580247,5.006329


In [4]:
# Cell 4 — Check for nulls
df.isnull().sum().sort_values(ascending=False).head(15)

Position              39300
LapStartDate          39300
PitInTime             28026
PitOutTime            27926
LapTime               12603
SpeedFL               11272
Sector1SessionTime     9434
Sector1Time            9431
Sector3Time            2820
Sector3SessionTime     2820
SpeedST                2252
SpeedI2                 647
Sector2Time             638
Sector2SessionTime      638
SpeedI1                 468
dtype: int64

In [5]:
df[["Driver", "Year", "EventName", "Compound", "TrackTemp", "AirTemp"]].isnull().sum()

Driver       0
Year         0
EventName    0
Compound     0
TrackTemp    0
AirTemp      0
dtype: int64

In [6]:
print("Seasons:", sorted(df["Year"].unique()))

Seasons: [2021, 2022, 2023, 2024, 2025, 2026]


In [9]:
# Cell — Check for missing rounds per season

import fastf1
import pandas as pd
import os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent

fastf1.Cache.enable_cache(str(PROJECT_ROOT / "data/raw/fastf1_cache"))

seasons_collected = sorted(df["Year"].unique())
missing_rounds_summary = []

for year in seasons_collected:
    # Get the official schedule for that season
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    expected_rounds = set(schedule["RoundNumber"].tolist())

    # Get what you actually have in your dataset
    collected_rounds = set(df[df["Year"] == year]["RoundNumber"].unique())

    missing = sorted(expected_rounds - collected_rounds)

    if missing:
        missing_events = schedule[schedule["RoundNumber"].isin(missing)][["RoundNumber", "EventName"]]
        for _, row in missing_events.iterrows():
            missing_rounds_summary.append({
                "Year": year,
                "RoundNumber": row["RoundNumber"],
                "EventName": row["EventName"]
            })

missing_df = pd.DataFrame(missing_rounds_summary)

if missing_df.empty:
    print("✅ No missing rounds — every season is fully collected.")
else:
    print(f"⚠️ Found {len(missing_df)} missing rounds:")
    print(missing_df.to_string(index=False))

⚠️ Found 13 missing rounds:
 Year  RoundNumber                EventName
 2024           21     São Paulo Grand Prix
 2026           11     Hungarian Grand Prix
 2026           12         Dutch Grand Prix
 2026           13       Italian Grand Prix
 2026           14       Spanish Grand Prix
 2026           15    Azerbaijan Grand Prix
 2026           16     Singapore Grand Prix
 2026           17 United States Grand Prix
 2026           18   Mexico City Grand Prix
 2026           19     São Paulo Grand Prix
 2026           20     Las Vegas Grand Prix
 2026           21         Qatar Grand Prix
 2026           22     Abu Dhabi Grand Prix
